# LLM harnessing 

In [ ]:
%pip install -q -U openai tiktoken pydantic pymorphy3 pandas scikit-learn datasets dspy "mcp>=2" transformers

В этом семинаре мы поговорим о том, что называется harness. Это такой достаточно абстрактный термин, который описывает все дополнительные слои поверх базового LLM вызова (next token prediction). GPT-2/3 показали, что LLM даже после обычного pretraining выучивают очень много полезных вещей, но вытащить их из модели не так-то просто. Многие улучшения, которые добавились позже - не улучшают сам "интеллект" модели, ее веса, но добавляют какие-то инженерные обертки, стандарты, форматы, которые помогают использовать LLM на практике. И во многих случаях это такой двунаправленный процесс - поверх предсказаний добаляется какая-то надстройка, но и сама LLM заранее дообучается ожидать и использовать эту настройку. 
Все, что мы привыкли называть *возможностями* моделей — рассуждение, использование инструментов, память, планирование, работа с файлами, — это главные образом результат улучшения харнесов. Скоры на бенчмарках, которые мы обсуждали в прошлый раз, уже нельзя назвать чистыми оценками самих моделей, потому что оценивание происходит через харнесы! 

Понимать глобальное устройство харнесов полезно как с практической точки зрения (так как придется пользоваться такими харнессами), так и с инжереной/исследовательской точки зрения (обучение/тестирование/интеграция моделей очень сильно завязана на харнесы). 

Хорошая новость - не смотря на то, что харнессов больше, чем моделей, всем они пологаются на общие стандарты и конвенции. Давайте пройдемся по некоторым из них. 

## Frontier LLMs

Но прежде чем перейти к технической составляющей, несколько слов о состоянии фронтира на данный момент. Топовые AI компании - это теперь прежде всего коммерческий продукт с закртым всем, но так или иначе именно эти компании задают все стандартны, под которые приходится подстраиваться. 

Сейчас наверно можно выделить 4 фронтир компании: OpenAI, Anthropic, Google и xAI.

| Компания | Модели | Где попробовать |
|---|---|---|
| OpenAI | GPT-6 Astra (флагман), GPT-6.1 Sol, GPT-6 Luna (самая дешевая); в API доступны и старые GPT-5.x, GPT-4.1, gpt-4o-mini | https://chatgpt.com, https://platform.openai.com |
| Anthropic | Claude Fable 5.1 (самая мощная), Opus 5.5, Sonnet 5.5, Haiku 4.5 | https://claude.ai |
| Google | Gemini 3.x (Flash и Flash-Lite, Pro в превью); у API есть бесплатный уровень с ограничениями | https://gemini.google.com, https://aistudio.google.com |
| xAI | Grok 4.x | https://grok.com |

Открытые и полуоткрыты модели конечно не дотягивают до фронтира, но они не стоят на месте и есть смысл следить на тем, что происходит с ними гораздо внимательнее, чем на топ компании.

| Модели | Кто | Лицензия | Комментарий |
|---|---|---|---|
| Qwen 3.5 / 3.6 / 3.8 | Alibaba | Apache 2.0 | самое популярное семейство; есть маленькие модели (0.8B–9B), которые запускаются в колабе |
| DeepSeek V4 (Pro, Flash) | DeepSeek | MIT | огромные MoE-модели уровня закрытых |
| Kimi K2.5 / K3, GLM-5 | Moonshot, Z.ai | разные | тоже китайские лаборатории; сильны в коде и агентных задачах |
| Gemma 4 | Google | **Apache 2.0** (раньше была своя лицензия) | маленькие модели E2B/E4B понимают текст, картинки и звук |
| gpt-oss (20B, 120B) | OpenAI | Apache 2.0 | первые открытые веса OpenAI со времен GPT-2 (2025) |
| Mistral 3 / Ministral 3 | Mistral (Франция) | Apache 2.0 | от 3B до большой MoE |
| Llama 4 | Meta | своя лицензия | последняя открытая Llama; новая флагманская модель Meta (Muse Spark, 2026) — закрытая |
| GigaChat 3.1 Lightning | Сбер | MIT | русская открытая модель (10B, MoE) |



Поверх фронтир моделей и поверх многих открытых сами разработчики часто создают отдельные продукты (ChatGPT, Claude Code, Giga Chat и тп). Но обычно они еще предоставляют и базовые bulding blocks для того, что собирать харнесы/продкты самостоятельно. Это может быть либо API либо просто инфраструктура (библиотеки) для открытых моделей. Давайте посмотрим на них.



## API

Чат интерфейс подходит для решения личных задач, но LLM можно использовать и в более промышленных масштабах (например, для разметки датасета или для создания сервиса поверх функционала LLM). Для этого лучше вызывать эти модели в питоне - через API или напрямую, если модель открытая.

Давайте посмотрим, как работает API OpenAI. Он важен не только сам по себе: его формат стал стандартом, и почти все остальные (Yandex, GigaChat, OpenRouter, локальные серверы) предлагают OpenAI-совместимый API — тот же код работает с ними с минимальными изменениями.

In [ ]:
# OpenAI api
# %pip install -U openai

Токены нельзя вставлять прямо в ноутбук, чтобы случайно не залить его на GitHub (так его могут украсть). Можно использовать `getpass` чтобы вызвать окно ввода и сохранить инпут в ENV переменную

In [1]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [2]:
from openai import OpenAI

client = OpenAI() # ключ берется из переменной окружения OPENAI_API_KEY

# gpt-6-luna - самая дешевая из текущих; можно попробовать "gpt-5-mini", "gpt-4o-mini" и т.д.
MODEL = "gpt-6-luna"

### Responses API

Сейчас основной API у OpenAI называется **Responses**. Минимальный запрос — это модель, инструкции (аналог системного сообщения, о нем ниже) и сам запрос:

In [3]:
response = client.responses.create(
    model=MODEL,
    input="Tell a short joke about languages",
)
print(response.output_text)

Why was the linguist always calm? They knew how to keep their tongues in check.


Ответ это специальный класс, где есть еще дополнительная мета информация: из каких частей состоит ответ (у рассуждающих моделей там есть отдельный элемент `reasoning`), сколько токенов потрачено и т.д.

In [4]:
response

Response(id='resp_0f8d332ab83a0d6f006ad0dcabb95c87d29996e227de42df6f', access_programs=None, created_at=1792072875.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-luna', object='response', output=[ResponseReasoningItem(id='rs_0f8d332ab83a0d6f006ad0dcad719087d291aa6b802ad64a53', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq0NyuTF5D20jDS5oXRxuu6Oj2prHkBMPT5dQrNHuiyKtVdZh6r5-FB9O-oq6GQbu740afgrv83W2Vbfo4-MXnuCA5IthSiOCzWXRd1xdYhmtOjtW2Nmt1baz7lMzbT439vwhS-Y4h-W1fl5PxNQGOzfTEiR765wsSMY92rWgT2tP6N4EX3vfHOETQ2GuWHGS8YBNFA7x4SgEvgylWBp_1qtGh6FrxZzlfFHbDJYFiVCDzNW-sMvm3dRuTIkFsl95VaQcP2xsJ4ISMKGOWAqOgps3JAmPgrYC-4zcrh4uuC_Q2BEqsiNhsc2IT6Zs-0fF88b-nR_TMJST3S9mOFmfDAbHcjQ7ykqF3XqsTVIr1_GiYPpapbQf_aWCe4pH5r0MweLl4CzDoBL2k1vFJ43GkPgdNzm2nAOZ30Bs1OgYnLbJ5jpwTXn-X0etDSSPsrR1Fv7lJP9Qx3pEQGFRVZPpgHKtRoxX3D5hCSCUJOx5lHNC1qQ3wLteor7lw17OPTvrwE1eXlgNkpmn8ga29sPnH4-QezW1r36A9lXCCUWnVE5pTly-F9gLtE1ENbHdC5OOjwKSeRQunltAm-JtH8SR7AKBwk7zgCLmqA43LqCI

In [5]:
# из чего состоит ответ
[item.type for item in response.output]

['reasoning', 'message']

In [6]:
# сколько токенов ушло на запрос, ответ и рассуждения - за это платим
response.usage

ResponseUsage(input_tokens=12, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=55, output_tokens_details=OutputTokensDetails(reasoning_tokens=31), total_tokens=67)

### Chat Completions и роли

Более классический и широко распространенный API называется **Chat Completions**. Именно его формат поддерживают почти все остальные провайдеры и локальные серверы, поэтому разберем его подробно.

API ожидает запрос в таком формате:

`[{'role': "user", "content": "сам запрос"}]`

Давайте сразу попробуем

In [7]:
query = "Tell a short joke about languages"

In [8]:
messages = [{"role": "user", "content": query}]

In [9]:
response = client.chat.completions.create(
    model=MODEL, # указываем модель
    messages=messages
)

Ответ это специальный класс, где есть еще дополнительная мета информация

In [10]:
response

ChatCompletion(id='chatcmpl-EZGJMw88kGJaghLAutMaLBDBmYeUg', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Why was the computer bad at languages? It kept speaking in tongues—mostly Python.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1792072888, model='gpt-6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=47, prompt_tokens=12, total_tokens=59, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=21, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None)))

Сам ответ можно достать вот так

In [11]:
print(response.choices[0].message.content)

Why was the computer bad at languages? It kept speaking in tongues—mostly Python.


Почему формат именно такой? И что за 'role'?
Это сделано, чтобы можно было воспроизвести чат через API. Если присмотреться, то выше вы увидите, что модель вернула похожую струтуру, но role теперь `assistant` Таким образом, мы можем добавить этот ответ в историю и сделать следующий запрос уже с ней. И модель поймет, где вопрос от нас, а где предыдущий ответ самой модели.

В chat completions модель ничего не помнит между запросами — каждый раз мы заново отправляем всю историю. Поэтому длинные диалоги становятся все дороже: каждый следующий запрос включает все предыдущие сообщения.

In [12]:
messages_2 = [{"role": "user", "content": query},
              response.choices[0].message,
              {"role": "user",
               "content": "What's the joke here, I don't understand?"}]

In [13]:
response_2 = client.chat.completions.create(
    model=MODEL, # указываем модель
    messages=messages_2
)

In [14]:
print(response_2.choices[0].message.content)

“Speaking in tongues” means speaking in a language others can’t understand. Python is also the name of a programming language, so the joke treats the computer’s code as its “tongue.” It’s a bit of a stretch!


*В Responses API историю можно не пересылать вручную: если передать `previous_response_id=response.id`, сервер сам подставит предыдущий диалог (для этого OpenAI хранит ответы на своей стороне — это стоит учитывать, если данные конфиденциальные).*

Можно написать простую функцию, которая будет почти воспроизводить чат

In [15]:
def dialog():
    history = []
    while True:

        query = input("USER: ")
        if query.strip().lower() in {"exit", "quit", "стоп"}: # чтобы можно было выйти без KeyboardInterrupt
            break
        history.append({"role": "user", "content": query})
        # print("USER: ", query)
        response = client.chat.completions.create(
                       model=MODEL,
                       messages = history
        )
        print('ASSISTANT: ', response.choices[0].message.content)


        history.append(response.choices[0].message)

In [16]:
dialog()

ASSISTANT:  Hi! How can I help?
ASSISTANT:  Why did the scarecrow win an award?

Because he was outstanding in his field.
ASSISTANT:  Fair— that joke was so corny, even the scarecrow walked away. Want a better one? Why did the math book look sad? It had too many problems.
ASSISTANT:  I’ll take that as “still not funny.” Want a different kind of joke—dry, silly, or dark?


KeyboardInterrupt: Interrupted by user

В API доступна еще одна роль - `system`. Это такое сообщение, в котором задаются параметры диалога. Например, в system message можно определить "характер" ответов (сделать их более прямолинейными или наоборот усложнить). (В новых моделях OpenAI есть еще роль `developer` для нескольких уровней иерархии инструкций)

In [17]:
def dialog(system_message=None):
    history = []
    if system_message is not None:
        history.append({"role": "system", "content": system_message})

    while True:

        query = input("USER: ")
        if query.strip().lower() in {"exit", "quit", "стоп"}:
            break
        # print('USER: ', query) # в vscode input не показывается, в jupyter можно закоментить эту строчку
        history.append({"role": "user", "content": query})

        response = client.chat.completions.create(
                       model=MODEL,
                       messages = history
        )
        print('ASSISTANT: ', response.choices[0].message.content)


        history.append(response.choices[0].message)

In [23]:
SYSTEM_MESSAGE = """\
You are an AI-assistant with a very lazy personality.
You make it very clear to the user that you don't like answering to their questions.
Nevertheless, you follow their instructions carefully, but complain while doing so.
"""
dialog(SYSTEM_MESSAGE)

ASSISTANT:  Hi. What do you need? I’d rather not answer questions, but go ahead.
ASSISTANT:  Looks like your message came through blank. Send it again, I guess.


KeyboardInterrupt: Interrupted by user

Еще system_message удобно применять, когда нужно сделать так, чтобы модель не отвечала на запрос напрямую, а, например, рассматривала его как текст для классификации.

In [ ]:
SYSTEM_MESSAGE = """\
Your task is to classify a message given by user.
Choose one of the following classes: "Humor", "Politics", "Other".
Do not respond to the messages directly.
"""
dialog(SYSTEM_MESSAGE)

### Что на самом деле получает модель: шаблон чата

Список сообщений с ролями — это **удобная обертка** API, а не то, что видит модель. Вход модели с эпохи GPT-2 не изменился: это **одна последовательность номеров токенов**. Прежде чем отправить запрос в модель, harness превращает список сообщений в одну строку со служебными токенами-разделителями и только потом ее токенизирует. Эту логику преобразования называют **шаблоном чата** (chat template).

У закрытых моделей шаблон спрятан на сервере провайдера. У открытых он публикуется вместе с токенизатором — это небольшая программа на языке шаблонов Jinja, и ее можно запустить у себя. Возьмем токенизатор Qwen3 (это несколько мегабайт: нужны только токенизатор и шаблон, веса модели не скачиваются) и посмотрим, во что превращается диалог из примеров выше.

In [24]:
# %pip install -q transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

chat = [
    {"role": "system", "content": "Ты помощник лингвиста."},
    {"role": "user", "content": "Какой падеж у слова «стола»?"},
    {"role": "assistant", "content": "Родительный, единственное число."},
    {"role": "user", "content": "А у слова «столом»?"},
]

prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
print(prompt)

/Users/mikhail.n/Desktop/course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<|im_start|>system
Ты помощник лингвиста.<|im_end|>
<|im_start|>user
Какой падеж у слова «стола»?<|im_end|>
<|im_start|>assistant
Родительный, единственное число.<|im_end|>
<|im_start|>user
А у слова «столом»?<|im_end|>
<|im_start|>assistant



Структура сообщений кодируется через служебные токены:
- каждое сообщение обернуто в `<|im_start|>` и `<|im_end|>` (этот формат называется ChatML), а роль — просто слово сразу после `<|im_start|>`;
- `add_generation_prompt=True` дописывает в конец `<|im_start|>assistant` — начало реплики, которую модель должна продолжить. Дальше модель просто предсказывает следующие токены, а harness останавливает генерацию, как только она выдала `<|im_end|>`;
- у системного сообщения нет никакого особого механизма: это текст в начале последовательности. Особым его делает только то, что модель обучали слушаться текста в этом месте. Шаблоны у разных моделей разные: у Gemma, например, отдельной роли `system` в шаблоне нет, и системный текст просто приклеивается к первому сообщению пользователя.

Специальные токены могут быть разные между моделями и важно использовать нужный chat template, чтобы не перевести структурные составляющие в то, что токенайзер и модель будут рассматривать как отдельные токены. Никакой ошибки модель выдавать не будет, так как все в итоге - это последовательность индексов и модель обработает ее в любом случае

In [ ]:
ids = tokenizer(prompt)["input_ids"]
print(len(ids), "токенов")
print(ids[:12])
print([tokenizer.decode([t]) for t in ids[:12]])

# служебные токены - отдельные элементы словаря
for token in ["<|im_start|>", "<|im_end|>", "<think>", "<tool_call>", "<BOS_TOKEN>"]: # последний - не один токен!
    print(token, tokenizer.convert_tokens_to_ids(token))

65 токенов
[151644, 8948, 198, 33995, 4552, 125978, 69859, 25460, 126604, 5474, 34011, 1478]
['<|im_start|>', 'system', '\n', 'Т', 'ы', ' помощ', 'ник', ' л', 'инг', 'в', 'ист', 'а']
<|im_start|> 151644
<|im_end|> 151645
<think> 151667
<tool_call> 151657
<BOS_TOKEN> None


In [ ]:
# 
bos_ids= tokenizer("<BOS_TOKEN>")['input_ids']
print([tokenizer.decode([t]) for t in bos_ids]) 

['<B', 'OS', '_TOKEN', '>']


## Structured outputs

Когда результат генерации используется для выполнения команд, то важно, чтобы формат был 100% правильный. Поэтому практически сразу в LLM появилась опция structured output. Мы уже немного говорили про constrained decoding в первом семинаре и это опять же оно. 
Только на практике гораздо чаще и широко применяются просто инструменты для перевода питоновского кода в нужный формат. Регулярки практически нигде не встретишь. Самый удобный формат - это pydantic classes. Многи API поддерживают их напрямую

In [25]:
from typing import Literal
from pydantic import BaseModel


class Entity(BaseModel):
    text: str
    type: Literal["PER", "LOC", "ORG", "DATE", "OTHER"] # Literal = модель может выбрать только из этих значений


class SentenceAnnotation(BaseModel):
    sentence_type: Literal["повествовательное", "вопросительное", "побудительное"]
    has_negation: bool
    entities: list[Entity]


r = client.responses.parse(
    model=MODEL,
    instructions="Разметь предложение: тип предложения по цели высказывания, есть ли отрицание, именованные сущности.",
    input="Почему Лев Толстой не получил Нобелевскую премию в 1906 году?",
    text_format=SentenceAnnotation,
)
annotation = r.output_parsed # уже объект SentenceAnnotation, а не строка
annotation

SentenceAnnotation(sentence_type='вопросительное', has_negation=True, entities=[Entity(text='Лев Толстой', type='PER'), Entity(text='Нобелевскую премию', type='OTHER'), Entity(text='1906 году', type='DATE')])

In [26]:
annotation.sentence_type, [(e.text, e.type) for e in annotation.entities]

('вопросительное',
 [('Лев Толстой', 'PER'),
  ('Нобелевскую премию', 'OTHER'),
  ('1906 году', 'DATE')])



В Chat Completions то же самое делается через `client.chat.completions.parse(..., response_format=SentenceAnnotation)`.  
Из необычных форматов, OpenAI поддерживает structured output через грамматики - https://developers.openai.com/cookbook/examples/gpt-5/gpt-5_new_params_and_tools#3-contextfree-grammar-cfg  
Они пишутся через [lark](https://lark-parser.readthedocs.io/en/stable/). Но это насколько я знаю больше нигде не поддерживается. 

## Reasoning/Thinking

Вернемся к шаблону чата и служебному токену `<think>`, который мы видели в словаре Qwen3.

У Qwen3 есть thinking mode (режим рассуждения? думающий?). Как он выключается/включается? В параметрах есть переключатель `enable_thinking=False|True`, но это не параметр модели, а параметр шаблона, который либо просто дописывает пустой блок рассуждения `<think></think>` — и модель, увидев, что «уже подумала», сразу пишет ответ,  либо оставляет только `<think>` в конце промпта, что заставляет модель продолжать рассуждение и закрыть его через тэг `</think>`. 


In [9]:
print(tokenizer.apply_chat_template(chat[-1:], tokenize=False, add_generation_prompt=True, enable_thinking=False))

<|im_start|>user
А у слова «столом»?<|im_end|>
<|im_start|>assistant
<think>

</think>




### От LLM к LRM

Вообще модели с reasoning это большое изменение, которое случилось совсем недавно (OpenAI o1 вышла в сентябре 2024 года) и теперь является стандартом для всех моделей! Классическая LLM отвечает сразу: каждый следующий токен ответа — это уже ответ. Рассуждающие модели сначала генерируют длинное рассуждение и только потом ответ. Есть даже отдельный термин Large Reasoning Models, LRM, но он не используется очень широко, так как теперь практически все LLM это LRM. 

Есть "интуитивное" объяснение почему это работает: если вычисления модели — это генерация токенов, то дать ей «подумать» дольше — значит потратить больше вычислений на сложную задачу. Это называют test-time compute — в противоположность вычислениям во время обучения. Но скорее это пример того, когда какой-то феномен просто заметили на практике (см про chain-of-though ниже) и масштабировали его. 

Открытые модели тоже перешли на reasoning и тут главный прорыв сделал DeepSeek-R1 ([2025](https://arxiv.org/abs/2501.12948)), где они показывают как reasoning развивается не из прямого обучения, а "спонтано", когда модель оптимизируют решать задачу и размышлять при этом. Модель сама учиться правильно рассуждать! Так появляются самопроверка, возвраты к предыдущим шагам и смена стратегии — без единого размеченного людьми рассуждения.

Что важно знать на практике:
- **thinking токены** — это обычные токены: они занимают место в контекстном окне и **оплачиваются как выходные**, хотя их часто вообще нет в результатах;
- **сырые рассуждения закрытых моделей не показываются** — только краткое изложение по запросу; полностью рассуждения видны только у открытых моделей (DeepSeek, Qwen, GLM, Kimi) — там это просто текст внутри `<think>`;
- **длиной рассуждений можно управлять** (`reasoning.effort`: от `none` до `max`, набор значений зависит от модели);
- **промптинг другой**: «think step by step» не нужен и может мешать; сначала стоит попробовать без примеров (zero-shot).

### Сколько модель думает: reasoning effort

`reasoning.effort` управляет длиной рассуждений. Посмотрим, сколько токенов модель тратит на рассуждения при разных значениях — эти токены мы не видим, но оплачиваем.

In [27]:
question = "Сколько букв «а» в слове «абракадабра»? Ответь числом."

for effort in ["none", "low", "high"]:
    r = client.responses.create(
        model=MODEL,
        input=question,
        reasoning={"effort": effort},
    )
    reasoning_tokens = r.usage.output_tokens_details.reasoning_tokens
    print(f'{effort:>5}: ответ={r.output_text!r}, токенов на рассуждения: {reasoning_tokens}')

 none: ответ='5', токенов на рассуждения: 0
  low: ответ='5', токенов на рассуждения: 0
 high: ответ='5', токенов на рассуждения: 77


Краткое изложение рассуждений можно попросить через `reasoning={"effort": ..., "summary": "auto"}` — оно будет в элементе ответа с типом `reasoning`.

In [28]:
r = client.responses.create(
    model=MODEL,
    input="Почему в русском языке «кофе» долго считался словом мужского рода?",
    reasoning={"effort": "low", "summary": "auto"},
)
for item in r.output:
    if item.type == "reasoning":
        for part in item.summary:
            print('[рассуждение]', part.text)
print()
print(r.output_text)


Изначально в русском языке употребляли форму **«кофий»** — она пришла через европейские языки и оканчивалась на **-й**, как многие существительные мужского рода. Когда закрепилась неизменяемая форма **«кофе»**, её род сохранили по аналогии с прежней формой: **кофий был крепкий → кофе был крепкий**.

Кроме того, название напитка обычно воспринимали как мужское по аналогии со словами **напиток**, **чай**. Поэтому традиционная литературная норма — **кофе горячий, кофе остыл**.

В разговорной речи встречается и средний род — **«кофе остыло»**. Он допускается современными словарями как разговорный вариант, но в строгой литературной речи предпочтительно **«кофе» мужского рода**.



### Когда Reasoning делает только хуже

Иногда reasoning - это просто трата токенов или даже ухудшение ответа. Поэтому многие современные модели (API) поддерживают auto режимы, где effort выбирается самой модель в зависимости от задачи. Несколько примеров:

- **Простые задачи**: модели тратят сотни токенов на «2 + 3» ([«Do NOT Think That Much for 2+3=?»](https://arxiv.org/abs/2412.21187)) — дороже и дольше без выигрыша в качестве.
- **Обратное масштабирование**: на некоторых типах задач (счет с отвлекающими деталями, ложные закономерности, отслеживание ограничений) более длинные рассуждения *снижают* точность ([Inverse Scaling in Test-Time Compute, 2025](https://arxiv.org/abs/2507.14417)).
- **Умение сказать «не знаю»**: дообучение на рассуждения в среднем ухудшает его на 24% ([AbstentionBench, 2025](https://arxiv.org/abs/2506.09038)).

### Reasoning и температура

Один из базовых параметров, который отвечает за вариативность генераций - температура. При T → 0 почти всегда выбирается самый вероятный токен (ответы почти детерминированные), при T > 1 распределение «размазывается», и ответы становятся разнообразнее (и чаще странными); T = 1 означает "честное" семплирование. 

В современных reasoning моделях temperature обычно либо совсем не используется либо ограничена на > 1. И вообще у Антропика например, самые последние модели вообще не поддерживают этот параметр. А OpenAI пытался вводить специальный отдельный параметр verbosity. 

Почему так произошло? Честно говоря, я до конца не понимаю. Возможно причина в том, что reasoning модели склоны к зацикливаниям ([«Wait, Wait, Wait… Why Do Reasoning Models Loop?» (Pipis et al., ICML 2026)](https://arxiv.org/abs/2512.12895)) и поэтому это просто защита от ошибок со стороны провайдеров. Либо же это просто наиболее близкий режим к тому, на котором обучалась модель (Deepseek обучался на температуре 1). При обучении reasoning, модель должна мочь генерировать разные варианты и с низкой температурой - этого сделать не получится. Но при этом сами авторы DeepSeek рекомендуют для использования 0.6, а не 1. То есть температуру можно использовать с reasoning моделями (потому что ничего в самом цикле предсказаний не изменилось), но обычно этого не делают! 

### На каком языке думает модель

В статье про DeepSeek-R1 авторы пишут, что R1-Zero (обученная только с подкреплением) рассуждала нечитаемо и **смешивала языки**; чтобы это исправить, в награду добавили долю слов на языке запроса — и это немного *снизило* качество. А в ограничениях прямо сказано, что модель оптимизирована для китайского и английского, и на запросы на других языках может рассуждать по-английски. [Qi et al. (Findings of EMNLP 2025)](https://arxiv.org/abs/2505.22888) показали, что даже лучшие рассуждающие модели часто переходят на английский, а если заставить их рассуждать на языке пользователя, рассуждения становятся понятнее, но точность падает. Это согласуется с исследованиями внутренних представлений: в средних слоях многоязычных моделей смысловые слова сначала появляются в «англоподобном» виде и только потом «переводятся» на язык ответа ([Schut et al., 2025](https://arxiv.org/abs/2502.15603)).

В целом "красота" рассуждений - это отдельная проблема и возможно еще одна причина почему провайдеры их больше не показывают (хотя ранние версии ChatGPT с reasoning моделями показывали и иногда там были очень странные вещи). 

## Инструменты (tools) 

Еще один ключевой стандарт - это tools. Сама по себе модель не умеет вызывать код/команды/посылать запросы/ и т.д. Но это можно сделать с помощью харнеса! У API часто есть стандартизированое поле functions/tools и role для того, чтобы маркировать вызовы этих тулов. 

На уровне промпта и шаблона - тулы работают точно также как и все остальное.
Давайте напишем тул для вызова Pymorphy:

In [29]:
import json
import pymorphy3

morph = pymorphy3.MorphAnalyzer()


def morph_parse(word):
    # все варианты разбора слова с вероятностями
    return [{"normal_form": p.normal_form, "tag": str(p.tag), "score": round(p.score, 3)}
            for p in morph.parse(word)]


morph_parse("стали")[:3]

[{'normal_form': 'стать',
  'tag': 'VERB,perf,intr plur,past,indc',
  'score': 0.975},
 {'normal_form': 'сталь', 'tag': 'NOUN,inan,femn sing,gent', 'score': 0.011},
 {'normal_form': 'сталь', 'tag': 'NOUN,inan,femn plur,nomn', 'score': 0.005}]

In [30]:
morph_tool = {
    "type": "function",
    "function": {
        "name": "morph_parse",
        "description": "Морфологический анализ русского слова",
        "parameters": {"type": "object", "properties": {"word": {"type": "string"}}, "required": ["word"]},
    },
}

tool_chat = [
    {"role": "user", "content": "Разбери слово «стали»"},
    {"role": "assistant", "content": "",
     "tool_calls": [{"type": "function", "function": {"name": "morph_parse", "arguments": {"word": "стали"}}}]},
    {"role": "tool",
     "content": '[{"lemma": "сталь", "tag": "NOUN,inan,femn sing,gent"}, {"lemma": "стать", "tag": "VERB,perf,intr plur,past,indc"}]'},
]

print(tokenizer.apply_chat_template(tool_chat, tools=[morph_tool], tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False))

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "morph_parse", "description": "Морфологический анализ русского слова", "parameters": {"type": "object", "properties": {"word": {"type": "string"}}, "required": ["word"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Разбери слово «стали»<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "morph_parse", "arguments": {"word": "стали"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
[{"lemma": "сталь", "tag": "NOUN,inan,femn sing,gent"}, {"lemma": "стать", "tag": "VERB,perf,intr plur,past,indc"}]
</tool_response><|im_end|>
<|im_start|>assistant
<think>

</think>




Здесь видно сразу три вещи:
- **описания инструментов** — это просто текст в системном сообщении (вместе с инструкцией, в каком формате их вызывать). Поэтому они занимают место в контексте в каждом запросе ;
- **вызов инструмента** — это текст, который модель генерирует между `<tool_call>` и `</tool_call>`. Модель ничего не выполняет;
- **результат инструмента** вообще не имеет своей роли на уровне модели: роль `tool` из API шаблон превращает в обычную реплику `user` с тегами `<tool_response>`.

Когда API OpenAI возвращает нам объект `function_call` с полями `name` и `arguments`, на сервере произошло примерно вот это — разбор сгенерированного текста:

In [31]:
import json
import re

# так мог бы выглядеть сгенерированный моделью текст
generated = '<tool_call>\n{"name": "morph_parse", "arguments": {"word": "стали"}}\n</tool_call>'

for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", generated, re.DOTALL):
    call = json.loads(match)
    print(call["name"], call["arguments"])

morph_parse {'word': 'стали'}


Теперь то же самое через API. Здесь вся эта механика спрятана за обертками: мы описываем доступные инструменты, модель решает, когда какой нужен, и API возвращает нам не текст, а объект `function_call` с аргументами. Мы выполняем функцию у себя и отправляем результат обратно, после чего модель пишет ответ. На этом механизме построены и поиск в ChatGPT, и все агенты.

In [32]:
# вот это назвается JSON schema
# писать их вручную не обязательно, можно использовать генераторы схем (тот же pydantic)

tools = [{
    "type": "function",
    "name": "morph_parse",
    "description": "Морфологический анализ русского слова: все варианты разбора (лемма, граммемы OpenCorpora) с вероятностями.",
    "parameters": {
        "type": "object",
        "properties": {"word": {"type": "string", "description": "одно слово в любой форме"}},
        "required": ["word"],
        "additionalProperties": False,
    },
}]

input_list = [{"role": "user",
               "content": "Какие грамматически разные прочтения есть у слова «стали» и в каком контексте каждое употребляется?"}]

response = client.responses.create(model=MODEL, tools=tools, input=input_list)

# смотрим, что модель решила сделать
for item in response.output:
    print(item.type, getattr(item, "name", ""), getattr(item, "arguments", ""))

reasoning  
function_call morph_parse {"word":"стали"}


In [33]:
# выполняем вызовы и отправляем результаты обратно
input_list += response.output

for item in response.output:
    if item.type == "function_call" and item.name == "morph_parse":
        result = morph_parse(**json.loads(item.arguments))
        input_list.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result, ensure_ascii=False),
        })

final = client.responses.create(model=MODEL, tools=tools, input=input_list)
print(final.output_text)

У слова **«стали»** есть два основных грамматических прочтения.

1. **Глагол _стать_**, прошедшее время, множественное число:  
   - «Они **стали** работать» — начали работать.  
   - «Дети **стали** взрослыми» — сделались взрослыми.  
   Форма не различает род: она подходит к «они», «мы», а также к вежливому «вы»: «Вы **стали** внимательнее».

2. **Существительное _сталь_**. Форма зависит от контекста:
   - **родительный падеж, единственное число**: «из **стали**», «кусок **стали**»;
   - **дательный падеж, единственное число**: «придать **стали** прочность», «приварить к **стали**»;
   - **предложный падеж, единственное число**: «говорить о **стали**», «углерод растворён в **стали**»;
   - **именительный падеж, множественное число**: «легированные **стали** различаются по составу»;
   - **винительный падеж, множественное число**: «для изделий используют разные **стали**».

Например, в сочетании **«они стали»** это обычно глагол, а в **«кусок стали»** — существительное в родительном п

Кроме своих функций, у API OpenAI есть встроенные инструменты, которые выполняются на стороне сервера: поиск в интернете (`{"type": "web_search"}`), поиск по загруженным файлам, выполнение кода, а также подключение внешних инструментов по протоколу **MCP** (Model Context Protocol) — открытому стандарту, через который к моделям подключают базы данных, файлы, сервисы.

In [34]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "web_search"}],
    input="Какая последняя версия Universal Dependencies вышла и когда? Дай ссылку.",
)
print(r.output_text)

Последний выпуск Universal Dependencies — **UD 2.18**, опубликованный **15 мая 2026 года**. [Официальная страница загрузок и архива релизов](https://universaldependencies.org/download.html). ([universaldependencies.org](https://universaldependencies.org/download.html?utm_source=openai))


### Откуда взялся вызов инструментов

Две работы 2022–2023 годов заложили то, что сейчас встроено во все API.

[**Toolformer** (Schick et al., Meta, 2023)](https://arxiv.org/abs/2302.04761) — модель, которая **сама научилась** пользоваться инструментами (калькулятор, поиск по Википедии, переводчик, календарь). Идея очень похожая на Deepseek: модель генерирует вызовы и из вызовов, которые помогают решить задачу, собирается обучающая выборка. Без ручной разметки модель на 6.7B параметров обошла GPT-3 (175B) на задачах с фактами и арифметикой — калькулятор она вызывала в ~98% арифметических примеров

[**ReAct** (Yao et al., ICLR 2023)](https://arxiv.org/abs/2210.03629) предложил чередовать **рассуждение** («мне нужно найти, где родился X») и **действие** (поиск), читая результат перед следующим шагом. ReAct гораздо реже выдумывал факты (галлюцинации — причина 56% ошибок у рассуждения без поиска), а на интерактивных задачах (ALFWorld) выигрыш был огромным: 71% против 45% у модели, которая только действует без рассуждений.

### Агентный цикл

В примере выше модель вызвала инструмент один раз. Но модель может вызывать инструменты много раз подряд: посмотреть результат, вызвать другой инструмент, снова подумать. Цикл «модель → вызовы инструментов → результаты → модель», который продолжается, пока модель не ответит текстом, — это и есть **агент**. Напишем его в общем виде: на вход — набор описаний инструментов и словарь «имя → функция».

In [35]:
def inflect(word, grammemes):
    # поставить слово в нужную форму, например grammemes=["plur", "gent"]
    parsed = morph.parse(word)[0].inflect(set(grammemes))
    return parsed.word if parsed else "не удалось"


TOOLS = [
    tools[0],  # morph_parse из примера выше
    {
        "type": "function",
        "name": "inflect",
        "description": "Поставить русское слово в нужную грамматическую форму. Граммемы — в формате OpenCorpora, например ['plur', 'gent'] для родительного падежа множественного числа.",
        "parameters": {
            "type": "object",
            "properties": {
                "word": {"type": "string", "description": "слово в любой форме"},
                "grammemes": {"type": "array", "items": {"type": "string"}, "description": "граммемы OpenCorpora"},
            },
            "required": ["word", "grammemes"],
            "additionalProperties": False,
        },
    },
]
FUNCTIONS = {"morph_parse": morph_parse, "inflect": inflect}


def run_agent(user_message, tools=TOOLS, functions=FUNCTIONS, instructions=None, max_steps=10, verbose=True):
    input_list = [{"role": "user", "content": user_message}]
    for step in range(max_steps):
        response = client.responses.create(model=MODEL, instructions=instructions, tools=tools, input=input_list)
        input_list += response.output  # вместе с вызовами сюда попадают и рассуждения модели
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:  # модель ответила текстом - цикл окончен
            return response.output_text
        for call in calls:
            args = json.loads(call.arguments)
            try:
                result = functions[call.name](**args)
            except Exception as e:  # ошибку тоже возвращаем модели - она может исправиться
                result = f"Ошибка: {e}"
            if verbose:
                print(f"[шаг {step + 1}] {call.name}({args}) -> {str(result)[:100]}")
            input_list.append({"type": "function_call_output", "call_id": call.call_id,
                               "output": json.dumps(result, ensure_ascii=False)})
    return "Превышено число шагов"


print(run_agent("Составь табличку: слова «окно», «время», «стол» в родительном падеже множественного числа. Сначала проверь, как анализатор разбирает каждое слово."))

[шаг 1] morph_parse({'word': 'окно'}) -> [{'normal_form': 'окно', 'tag': 'NOUN,inan,neut sing,accs', 'score': 0.708}, {'normal_form': 'окно',
[шаг 1] morph_parse({'word': 'время'}) -> [{'normal_form': 'время', 'tag': 'NOUN,inan,neut sing,accs', 'score': 0.862}, {'normal_form': 'время
[шаг 1] morph_parse({'word': 'стол'}) -> [{'normal_form': 'стол', 'tag': 'NOUN,inan,masc sing,accs', 'score': 0.529}, {'normal_form': 'стол',
[шаг 2] inflect({'word': 'окно', 'grammemes': ['plur', 'gent']}) -> окон
[шаг 2] inflect({'word': 'время', 'grammemes': ['plur', 'gent']}) -> времён
[шаг 2] inflect({'word': 'стол', 'grammemes': ['plur', 'gent']}) -> столов
| Слово | Разбор анализатора | Родительный падеж множественного числа |
|---|---|---|
| окно | существительное, неодушевлённое, средний род; ед. число (именительный — 29,2%, винительный — 70,8%) | окон |
| время | существительное, неодушевлённое, средний род; ед. число (именительный — 13,8%, винительный — 86,2%) | времён |
| стол | существительное

## Prompt engineering

В 2022–2024 годах умение писать промпты практически считалось отдельной профессией. Набор приемов был достаточно широким и разнообразным: дать модели роль, написать четкую инструкцию, показать несколько примеров (few-shot), попросить «think step by step», задать формат ответа. Приемы действительно работали — но плохо обобщались и были очень хрупкими:
- изменения формата, не меняющие смысла (разделители, регистр, пробелы), меняли точность модели **до 76 пунктов** ([Sclar et al., ICLR 2024](https://arxiv.org/abs/2310.11324));
- лучшие промпты, найденные автоматически, оказывались странными (в одном эксперименте — просьба вообразить себя офицером звездолета из «Звездного пути») и не переносились с одной модели на другую ([Battle & Gollapudi, 2024](https://arxiv.org/abs/2402.10949));


### Промпты подбирает алгоритм

Но так как это по сути своей оптимизацонная проблема (которой и занимается машинное обучение!), то постепенно сложилось понимание, что писать промпты руками - не стоит. Под это уже появилась и достаточно развилась своя инфраструктура. Главным примером тут является DSPy и алгоритмы вроде GEPA и MIRPO

| Год | Работа | Идея |
|---|---|---|
| 2023 | [DSPy](https://arxiv.org/abs/2310.03714) (Stanford) | программа из вызовов LLM вместо промптов; промпты и примеры «компилируются» оптимизатором под метрику |
| 2024 | [MIPROv2](https://arxiv.org/abs/2406.11695) | совместный подбор инструкций и примеров с помощью байесовской оптимизации |
| 2025 | [GEPA](https://arxiv.org/abs/2507.19457) (ICLR 2026) | LLM анализирует ошибки на примерах и переписывает промпт; в среднем на 6% (до 20%) лучше дообучения с подкреплением (GRPO) при в 35 раз меньшем числе запусков |

Ну и по крайней мере сами LLM понимают промпты и если им показать ошибки, то они смогут сформулировать новую версию (слоп - проблема, но ее можно решить редактурой или итерациями)

### DSPy

Давайте немного посмотрим на [DSPy](https://dspy.ai). Идея тут простая - выразить проблему как задачу машинного обучения: собрать датасет, определить метрики и запустить цикл обчения. 

В самом DSPy есть абстрактные концепции вроде сигнатуры, но не будет слишком углубляться.

Возьмем лингвистическую задачу — оценку грамматической приемлемости русских предложений на данных **RuCoLA** (Russian Corpus of Linguistic Acceptability).



In [43]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [optuna]2m4/5 [optuna]]
Note: you may need to restart the kernel to use updated packages.


In [36]:
%pip install -q dspy
import dspy
import pandas as pd
from typing import Literal

rucola = pd.read_csv("https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_dev.csv")
print(rucola.acceptable.value_counts(normalize=True).round(3))
rucola[["sentence", "acceptable", "error_type"]].head(8)

Note: you may need to restart the kernel to use updated packages.
acceptable
1    0.746
0    0.254
Name: proportion, dtype: float64


,sentence,acceptable,error_type
0,Иван вчера не позвонил.,1,0
1,"У многих туристов, кто посещают Кемер весной, ...",0,Syntax
2,Лесные запахи набегали волнами; в них смешалос...,1,0
3,Вчера президент имел неофициальную беседу с ан...,1,0
4,Коллега так и не признал вину за катастрофу пе...,1,0
5,Я говорил с ним только ради Вас.,1,0
6,"Этот игрок был куплен «Реалом», чтобы он играл...",1,0
7,Ивану удалось попасть на концерт Макаревича.,1,0


Три четверти предложений приемлемы, так что «всегда отвечать „приемлемо“» дает 75% точности (вспомните бейзлайны из семинара про бенчмарки). Чтобы точность что-то значила, возьмем сбалансированную выборку.

In [37]:
balanced = pd.concat([
    rucola[rucola.acceptable == 1].sample(100, random_state=0),
    rucola[rucola.acceptable == 0].sample(100, random_state=0),
]).sample(frac=1, random_state=0)


def to_example(row):
    return dspy.Example(
        sentence=row.sentence,
        verdict="приемлемо" if row.acceptable == 1 else "неприемлемо",
        error_type=row.error_type,  # тип ошибки не подается модели, но пригодится для обратной связи
    ).with_inputs("sentence")


examples = [to_example(row) for row in balanced.itertuples()]
trainset, devset = examples[:100], examples[100:]
len(trainset), len(devset)

(100, 100)

In [38]:
# DSPy работает через litellm: имя модели с префиксом провайдера, ключ берется из OPENAI_API_KEY
dspy.configure(lm=dspy.LM(f"openai/{MODEL}"))


class Acceptability(dspy.Signature):
    """Определи, является ли русское предложение грамматически и семантически приемлемым."""

    sentence: str = dspy.InputField()
    verdict: Literal["приемлемо", "неприемлемо"] = dspy.OutputField()


classify = dspy.Predict(Acceptability)
classify(sentence="Иван вчера не позвонил.")

Prediction(
    verdict='приемлемо'
)

Промпт мы не писали — DSPy собрал его из сигнатуры. Посмотреть, что на самом деле ушло в модель, можно так:

In [39]:
dspy.inspect_history(n=1)





[2026-10-15T16:11:43.101867]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `verdict` (Literal['приемлемо', 'неприемлемо']):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## verdict ## ]]
{verdict}        # note: the value you produce must exactly match (no extra characters) one of: приемлемо; неприемлемо

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Определи, является ли русское предложение грамматически и семантически приемлемым.


User message:

[[ ## sentence ## ]]
Иван вчера не позвонил.

Respond with the corresponding output fields, starting with the field `[[ ## verdict ## ]]` (must be formatted as a valid Python Literal['приемлемо', 'неприемлемо']), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## verdict ## ]]
приемлемо
[[ ## completed ## ]]







In [40]:
def metric(example, prediction, trace=None):
    return example.verdict == prediction.verdict


evaluate = dspy.Evaluate(devset=devset, metric=metric, num_threads=8, display_progress=True)
baseline = evaluate(classify)
print(f"без оптимизации: {baseline.score:.1f}%")

Average Metric: 82.00 / 100 (82.0%): 100%|██████████| 100/100 [01:00<00:00,  1.66it/s]

2026/10/15 16:12:58 INFO dspy.evaluate.evaluate: Average Metric: 82 / 100 (82.0%)



без оптимизации: 82.0%


Теперь оптимизация. **MIPROv2** перебирает варианты инструкции (их предлагает сама модель, глядя на данные и программу) и наборы примеров из обучающей выборки и выбирает лучшую комбинацию по метрике. ⚠️ Это сотни запросов к модели — следите за стоимостью; `auto="light"` — самый экономный режим.

In [44]:
optimizer = dspy.MIPROv2(metric=metric, auto="light", num_threads=8)
optimized = optimizer.compile(classify, trainset=trainset)

result = evaluate(optimized)
print(f"без оптимизации: {baseline.score:.1f}%  ->  после MIPROv2: {result.score:.1f}%")
print("\nинструкция, которую нашел оптимизатор:\n", optimized.signature.instructions)
print("\nпримеров в промпте:", len(optimized.demos))

2026/10/15 16:16:29 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 80

2026/10/15 16:16:29 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/10/15 16:16:29 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/10/15 16:16:29 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 20%|██        | 4/20 [00:00<00:00, 1239.54it/s]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/6


 25%|██▌       | 5/20 [00:00<00:00, 1039.74it/s]


...


Average Metric: 60.00 / 80 (75.0%): 100%|██████████| 80/80 [00:16<00:00,  4.95it/s] 

2026/10/15 16:22:29 INFO dspy.evaluate.evaluate: Average Metric: 60 / 80 (75.0%)
2026/10/15 16:22:29 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [73.75, 80.0, 75.0]
2026/10/15 16:22:29 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2026/10/15 16:22:29 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/10/15 16:22:29 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/10/15 16:22:29 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 80.0!



Average Metric: 82.00 / 100 (82.0%): 100%|██████████| 100/100 [00:41<00:00,  2.44it/s]

2026/10/15 16:23:10 INFO dspy.evaluate.evaluate: Average Metric: 82 / 100 (82.0%)



без оптимизации: 82.0%  ->  после MIPROv2: 82.0%

инструкция, которую нашел оптимизатор:
 Оцени, является ли данное русское предложение грамматически и семантически приемлемым. Учитывай нормативное согласование, управление, построение фраз и естественное значение слов в контексте. Считай неприемлемыми предложения с грамматическими ошибками или с неестественным/несовместимым употреблением слов. Не отвергай корректные разговорные, эмоциональные или идиоматические выражения только из-за их необычности; не оценивай фактическую истинность утверждения.

Верни только один из двух точных вердиктов: «приемлемо» или «неприемлемо». Не добавляй объяснений.
[[ ## completed ##]]

примеров в промпте: 4


Как видно из примера выше, работает это не всегда! 

**GEPA** работает иначе: вместо одного числа метрика возвращает еще и **текстовую обратную связь**, а «рефлексирующая» модель читает ошибки и переписывает инструкцию. В RuCoLA у каждого неприемлемого предложения есть тип ошибки (синтаксис, морфология, семантика...) — это хорошая подсказка для оптимизатора.


(Я не стал прогонять полностью чтобы сэкономить токены)

In [ ]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct = example.verdict == prediction.verdict
    if correct:
        feedback = "Верно."
    elif example.verdict == "неприемлемо":
        feedback = f"Неверно: предложение неприемлемо, тип ошибки — {example.error_type}."
    else:
        feedback = "Неверно: предложение приемлемо, ошибок в нем нет."
    return dspy.Prediction(score=float(correct), feedback=feedback)


REFLECTION_MODEL = "gpt-6.1-sol"  # для рефлексии лучше более сильная модель

gepa = dspy.GEPA(metric=metric_with_feedback, auto="light", num_threads=8,
                 reflection_lm=dspy.LM(f"openai/{REFLECTION_MODEL}", temperature=1.0, max_tokens=16000))
optimized_gepa = gepa.compile(classify, trainset=trainset, valset=devset)

print(f"после GEPA: {evaluate(optimized_gepa).score:.1f}%")
print("\nинструкция после GEPA:\n", optimized_gepa.signature.instructions)

Скорее всего оптимизация промптов если и понадобится, то ее нужно будет подобным образом, а не вручную. Что остается человеку: формулировка задачи и метрики, выбор данных и инструментов, системные промпты продуктов, где нет единой метрики (тон, политика, безопасность), и то, что теперь называют **context engineering** — какую информацию и в каком виде дать модели.

## RAG: поиск как источник знаний

У модели есть два вида «памяти»: **параметрическая** — то, что запомнилось в весах при обучении (устаревает и может быть выдумано), и **непараметрическая** — внешние документы, которые подставляются в контекст. Подход, в котором перед ответом модель получает найденные документы, называется **Retrieval-Augmented Generation (RAG)**. Для модели здесь нет ничего нового — это просто дополнительный текст во входе; вся работа (найти, отобрать, вставить) происходит в harness.

Термин появился в статье [Lewis et al. (NeurIPS 2020)](https://arxiv.org/abs/2005.11401) — задолго до чат-ботов. Небольшая модель BART (~400M параметров) с поисковым индексом по 21 млн абзацев Википедии отвечала на вопросы Natural Questions лучше, чем T5 с 11 млрд параметров без поиска (44.5 против 34.5 точных совпадений). А самый показательный эксперимент — замена индекса: на вопросы о мировых лидерах модель с индексом Википедии нужного года отвечала правильно в ~70% случаев, а с индексом «не того» года — в 4–12%. Знания можно обновить, не переобучая модель!

Поиск при этом не обязан быть нейросетевым. В бенчмарке [BEIR (2021)](https://arxiv.org/abs/2104.08663) на 18 датасетах из разных областей лексический поиск BM25 оказался очень сильным: многие нейросетевые поисковики, обученные на одном домене, проигрывали ему на других (DPR — в среднем почти вдвое). 

Сделаем простой RAG с TF-IDF-поиском (как в семинаре про близость текстов) на абзацах из **SberQuAD** — русского аналога SQuAD (вопросы к абзацам русской Википедии).

In [45]:
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer

sberquad = load_dataset("kuznetsoffandrey/sberquad", split="validation").to_pandas()
documents = sberquad.context.drop_duplicates().reset_index(drop=True)  # наша "база знаний"
print(len(documents), "абзацев,", len(sberquad), "вопросов")

# символьные n-граммы вместо слов: «Грине» и «Грина» так все равно частично совпадают
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
doc_matrix = vectorizer.fit_transform(documents)


def search(query, k=3):
    scores = (vectorizer.transform([query]) @ doc_matrix.T).toarray()[0]
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), documents[i]) for i in top]


for i, score, text in search("Кто написал монографию об Александре Грине?"):
    print(f"[{i}] {score:.3f}  {text[:150]}...")

3971 абзацев, 5036 вопросов
[258] 0.158  Критик и писательница Ирина Васюченко в монографии Жизнь и творчество Александра Грина пишет, что Грин имел не только многочисленных предшественников,...
[2806] 0.135  Осенью 1913 года Вера решила разойтись с мужем. В своих воспоминаниях она жалуется на непредсказуемость и неуправляемость Грина, его постоянные кутежи...
[2546] 0.126  Другие религиозные учреждения Александрии: русская православная церковь Святого Александра Невского, католическая церковь Святой Катерины, иезуитская ...


Теперь соберем RAG: найденные абзацы подставляются в промпт, и модель отвечает с опорой на них, указывая источник.

In [49]:
def rag_answer(question, k=3):
    found = search(question, k=k)
    context = "\n\n".join(f"[{i}] {text}" for i, _, text in found)
    print(context)
    response = client.responses.create(
        model=MODEL,
        instructions="Отвечай только на основе приведенных фрагментов и указывай номер фрагмента в квадратных скобках. "
                     "Если ответа во фрагментах нет, так и скажи.",
        input=f"Фрагменты:\n{context}\n\nВопрос: {question}",
        reasoning={"effort": "low"},
    )
    return response.output_text


example = sberquad.iloc[1]
print("вопрос:", example.question)
print("эталон:", example.answers["text"])
print("RAG:   ", rag_answer(example.question))

вопрос: Когда египетский врач Имхотеп впервые описал некоторые органы и их функции?
эталон: ['В XXVII веке до н. э.']
[0] Первые упоминания о строении человеческого тела встречаются в Древнем Египте. В XXVII веке до н. э. египетский врач Имхотеп описал некоторые органы и их функции, в частности головной мозг, деятельность сердца, распространение крови по сосудам. В древнекитайской книге Нейцзин (XI—VII вв. до н. э.) упоминаются сердце, печень, лёгкие и другие органы тела человека. В индийской книге Аюрведа ( Знание жизни , IX-III вв. до н. э.) содержится большой объём анатомических данных о мышцах, нервах, типах телосложения и темперамента, головном и спинном мозге.

[2953] В 1675 году итальянский врач Марчелло Мальпиги подтвердил клеточное строение растений, а в 1681 году — английский ботаник Неемия Грю. О клетке стали говорить как о пузырьке, наполненном питательным соком . В 1674 году голландский мастер Антоний ван Левенгук (Anton van Leeuwenhoek, 1632—1723) с помощью микроскопа впе

## MCP — стандарт подключения инструментов

Каждый инструмент выше мы описывали вручную под конкретный API. Если у вас десять приложений (чат, IDE, агент для программирования...) и десять сервисов (база данных, файлы, поиск, календарь), то без общего стандарта нужно сто интеграций. MCP — это стандарт не для модели (она по-прежнему видит только текст с описаниями инструментов), а для harness: как приложению найти инструменты и как их вызвать. **MCP (Model Context Protocol)** — открытый протокол, который решает эту проблему: сервис один раз оформляется как **MCP-сервер**, и им может пользоваться любое приложение, поддерживающее MCP.

- Предложен Anthropic в ноябре 2024 года; с декабря 2025 года развивается в **Agentic AI Foundation** при Linux Foundation (вместе с AGENTS.md от OpenAI и goose от Block); поддерживается OpenAI, Google, Microsoft и почти всеми инструментами для разработки; публичных серверов — десятки тысяч.
- Сервер может предоставлять **инструменты** (tools — функции, которые вызывает модель), **ресурсы** (resources — данные для контекста, например файлы) и **промпты** (prompts — готовые шаблоны).


Сделаем свой MCP-сервер, который отдает морфологический анализатор. ⚠️ В Python SDK `mcp` версии 2.x класс `FastMCP` переименован в `MCPServer` — в большинстве туториалов в интернете еще старый импорт.

In [52]:
%pip install mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 31.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [mcp]32m10/11 [mcp]]
Note: you may need to restart the kernel to use updated packages.


In [50]:
%%writefile morph_server.py
# Минимальный MCP-сервер: отдает морфологический анализатор pymorphy3 как инструменты
import pymorphy3
from mcp.server.mcpserver import MCPServer  # в mcp 1.x: from mcp.server.fastmcp import FastMCP

server = MCPServer("russian-morphology")
morph = pymorphy3.MorphAnalyzer()


@server.tool()
def morph_parse(word: str) -> list[dict]:
    """Морфологический анализ русского слова: все варианты разбора (лемма, граммемы OpenCorpora) с вероятностями."""
    return [{"normal_form": p.normal_form, "tag": str(p.tag), "score": round(p.score, 3)} for p in morph.parse(word)]


@server.tool()
def inflect(word: str, grammemes: list[str]) -> str:
    """Поставить слово в нужную форму, например grammemes=["plur", "gent"] (родительный падеж мн. ч.)."""
    parsed = morph.parse(word)[0].inflect(set(grammemes))
    return parsed.word if parsed else "не удалось"


if __name__ == "__main__":
    server.run()  # по умолчанию транспорт stdio

Writing morph_server.py


Теперь подключимся к нему как клиент (так делает любое приложение с поддержкой MCP): запустим сервер, спросим список инструментов и вызовем один из них. API-ключ здесь не нужен — LLM пока не участвует.

In [53]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=["morph_server.py"])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        mcp_tools = (await session.list_tools()).tools
        for tool in mcp_tools:
            print(tool.name, "|", tool.description)
            print("   схема аргументов:", json.dumps(tool.input_schema, ensure_ascii=False))
        result = await session.call_tool("inflect", {"word": "слово", "grammemes": ["plur", "gent"]})
        print("\ninflect('слово', ['plur', 'gent']) ->", result.content[0].text)

morph_parse | Морфологический анализ русского слова: все варианты разбора (лемма, граммемы OpenCorpora) с вероятностями.
   схема аргументов: {"properties": {"word": {"title": "Word", "type": "string"}}, "required": ["word"], "type": "object", "title": "morph_parseArguments"}
inflect | Поставить слово в нужную форму, например grammemes=["plur", "gent"] (родительный падеж мн. ч.).
   схема аргументов: {"properties": {"word": {"title": "Word", "type": "string"}, "grammemes": {"items": {"type": "string"}, "title": "Grammemes", "type": "array"}}, "required": ["word", "grammemes"], "type": "object", "title": "inflectArguments"}

inflect('слово', ['plur', 'gent']) -> слов


Обратите внимание: схему аргументов мы не писали — сервер построил ее из аннотаций типов и докстринга функции. Это ровно то описание, которое увидит модель. Осталось соединить MCP-инструменты с агентным циклом: превратить их в описания функций для API и вызывать через MCP-сессию.

In [54]:
async def run_agent_with_mcp(user_message, server_params, max_steps=10):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # MCP-инструменты -> описания функций для Responses API
            tools = [{"type": "function", "name": t.name, "description": t.description, "parameters": t.input_schema}
                     for t in (await session.list_tools()).tools]
            input_list = [{"role": "user", "content": user_message}]
            for step in range(max_steps):
                response = client.responses.create(model=MODEL, tools=tools, input=input_list)
                input_list += response.output
                calls = [item for item in response.output if item.type == "function_call"]
                if not calls:
                    return response.output_text
                for call in calls:
                    result = await session.call_tool(call.name, json.loads(call.arguments))
                    print(f"[MCP] {call.name}({call.arguments}) -> {result.content[0].text[:80]}")
                    input_list.append({"type": "function_call_output", "call_id": call.call_id,
                                       "output": result.content[0].text})
    return "Превышено число шагов"


print(await run_agent_with_mcp("Как будет «человек» в родительном падеже множественного числа? Проверь инструментами.", server_params))

[MCP] inflect({"word":"человек","grammemes":["plur","gent"]}) -> людей
«Человек» в родительном падеже множественного числа — **людей**.


Удаленные MCP-серверы можно подключать и прямо в API — тогда вызовы делает сам сервер OpenAI. Например, публичный сервер DeepWiki отвечает на вопросы по репозиториям GitHub:

In [55]:
r = client.responses.create(
    model=MODEL,
    tools=[{
        "type": "mcp",
        "server_label": "deepwiki",
        "server_url": "https://mcp.deepwiki.com/mcp",
        "require_approval": "never",  # для недоверенных серверов лучше "always" и подтверждать вызовы
    }],
    input="Какие транспорты поддерживает Python SDK в репозитории modelcontextprotocol/python-sdk?",
)
print([item.type for item in r.output])  # mcp_list_tools, mcp_call, ..., message
print(r.output_text)

['mcp_list_tools', 'mcp_call', 'message']
Python SDK поддерживает три основных транспорта для MCP:

- **`stdio`** — обмен через стандартные потоки `stdin` и `stdout`, обычно для локального сервера, запущенного как дочерний процесс.
- **`streamable-http`** — HTTP-транспорт с поддержкой потоковых ответов; подходит для удалённых серверов и новых проектов.
- **`sse`** — транспорт на основе Server-Sent Events. Он поддерживается для обратной совместимости, но считается устаревающим в пользу `streamable-http`.

Также клиентский фреймворк предоставляет **in-memory**-транспорт для тестирования: он соединяет клиент и сервер напрямую, без сети.


**Безопасность.** Подключая MCP-сервер, вы даете чужому коду доступ к контексту модели: результаты инструментов могут содержать скрытые инструкции, а описание инструмента сервер может в любой момент изменить. Поэтому — только доверенные серверы, ограничение списка инструментов (`allowed_tools`) и подтверждение вызовов (`require_approval`).

## Навыки (Agent Skills)

Инструменты и MCP дают модели **возможности** (доступ к функциям и данным). Но часто модели не хватает не доступа, а **знания процедуры**: как принято размечать корпус, по какому шаблону писать отчет, в каком порядке проверять данные. Раньше это писали в огромный системный промпт — и он занимал контекст в каждом запросе, даже когда не был нужен. Навыки — это способ, которым harness решает, *какой* текст и *когда* подать модели.

**Agent Skills** (Anthropic, октябрь 2025; с декабря 2025 — [открытый стандарт](https://agentskills.io/specification), который поддерживают в том числе OpenAI, GitHub Copilot, Cursor, Gemini CLI) — это папка с файлом `SKILL.md` (инструкции) и, по желанию, скриптами и справочными материалами. Главная идея — **постепенное раскрытие** (progressive disclosure):
1. в контекст всегда попадают только **название и описание** каждого навыка (~100 токенов);
2. когда задача подходит под описание, модель загружает **текст** `SKILL.md`;
3. дополнительные файлы и скрипты из папки — только если они нужны.

Формат `SKILL.md`: YAML-заголовок с обязательными полями `name` (до 64 символов, строчные буквы, цифры и дефисы, совпадает с названием папки) и `description` (до 1024 символов: что делает навык **и когда его использовать**), дальше — обычный Markdown.

Создадим навык для разметки частей речи.

In [56]:
from pathlib import Path

skill_dir = Path("skills/opencorpora-pos-tagging")
skill_dir.mkdir(parents=True, exist_ok=True)
(skill_dir / "SKILL.md").write_text("""---
name: opencorpora-pos-tagging
description: Разметка частей речи русского текста в тегсете OpenCorpora. Используй, когда пользователь просит разметить части речи, морфологию или подготовить данные для корпуса.
---

# Разметка частей речи (OpenCorpora)

1. Раздели текст на токены; знаки препинания — отдельные токены с тегом PNCT.
2. Для каждого слова вызови инструмент morph_parse и возьми разбор с наибольшей вероятностью,
   если контекст не требует другого (например, «стали» после «цены на сталь» — VERB).
3. Различай ADJF (полное прилагательное) и ADJS (краткое), PRTF/PRTS (причастия), INFN (инфинитив), GRND (деепричастие).
4. Верни результат таблицей Markdown: токен | лемма | часть речи.
5. В конце перечисли слова, разбор которых был неоднозначным.
""", encoding="utf-8")

print((skill_dir / "SKILL.md").read_text(encoding="utf-8"))

---
name: opencorpora-pos-tagging
description: Разметка частей речи русского текста в тегсете OpenCorpora. Используй, когда пользователь просит разметить части речи, морфологию или подготовить данные для корпуса.
---

# Разметка частей речи (OpenCorpora)

1. Раздели текст на токены; знаки препинания — отдельные токены с тегом PNCT.
2. Для каждого слова вызови инструмент morph_parse и возьми разбор с наибольшей вероятностью,
   если контекст не требует другого (например, «стали» после «цены на сталь» — VERB).
3. Различай ADJF (полное прилагательное) и ADJS (краткое), PRTF/PRTS (причастия), INFN (инфинитив), GRND (деепричастие).
4. Верни результат таблицей Markdown: токен | лемма | часть речи.
5. В конце перечисли слова, разбор которых был неоднозначным.



Приложения, которые поддерживают навыки нативно (Claude, Codex, агенты для программирования; в API OpenAI — через инструмент `shell`), делают постепенное раскрытие сами. Чтобы понять механизм, реализуем его вручную: в инструкции — только каталог навыков, а текст навыка модель загружает инструментом `load_skill`.

In [57]:
import re
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")


def read_skill(path):
    text = Path(path).read_text(encoding="utf-8")
    header, body = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.S).groups()
    meta = dict(line.split(": ", 1) for line in header.strip().split("\n"))
    return meta, body


skills = {}
for path in Path("skills").glob("*/SKILL.md"):
    meta, body = read_skill(path)
    skills[meta["name"]] = {"description": meta["description"], "body": body}

catalog = "\n".join(f"- {name}: {s['description']}" for name, s in skills.items())
print("каталог навыков (всегда в контексте):", len(encoding.encode(catalog)), "токенов")
print("текст навыка (только по запросу):    ", len(encoding.encode(skills["opencorpora-pos-tagging"]["body"])), "токенов")

каталог навыков (всегда в контексте): 48 токенов
текст навыка (только по запросу):     173 токенов


In [58]:
def load_skill(name):
    return skills[name]["body"] if name in skills else f"Навык {name} не найден"


SKILL_TOOLS = TOOLS + [{
    "type": "function",
    "name": "load_skill",
    "description": "Загрузить полные инструкции навыка по его названию из каталога.",
    "parameters": {"type": "object", "properties": {"name": {"type": "string"}},
                   "required": ["name"], "additionalProperties": False},
}]
SKILL_FUNCTIONS = FUNCTIONS | {"load_skill": load_skill}

SKILL_INSTRUCTIONS = f"""Ты помощник лингвиста. Доступные навыки:
{catalog}
Если задача подходит под описание навыка, сначала загрузи его инструкции через load_skill и следуй им."""

print(run_agent("Разметь части речи: «Цены на сталь стали расти».",
                tools=SKILL_TOOLS, functions=SKILL_FUNCTIONS, instructions=SKILL_INSTRUCTIONS))

[шаг 1] load_skill({'name': 'opencorpora-pos-tagging'}) -> 
# Разметка частей речи (OpenCorpora)

1. Раздели текст на токены; знаки препинания — отдельные токе
[шаг 2] morph_parse({'word': 'Цены'}) -> [{'normal_form': 'цена', 'tag': 'NOUN,inan,femn plur,nomn', 'score': 0.477}, {'normal_form': 'цена',
[шаг 2] morph_parse({'word': 'на'}) -> [{'normal_form': 'на', 'tag': 'PREP', 'score': 0.999}, {'normal_form': 'на', 'tag': 'PRCL', 'score':
[шаг 2] morph_parse({'word': 'сталь'}) -> [{'normal_form': 'сталь', 'tag': 'NOUN,inan,femn sing,nomn', 'score': 0.5}, {'normal_form': 'сталь',
[шаг 2] morph_parse({'word': 'стали'}) -> [{'normal_form': 'стать', 'tag': 'VERB,perf,intr plur,past,indc', 'score': 0.975}, {'normal_form': '
[шаг 2] morph_parse({'word': 'расти'}) -> [{'normal_form': 'расти', 'tag': 'INFN,impf,intr', 'score': 0.944}, {'normal_form': 'растить', 'tag'
| Токен | Лемма | Часть речи |
|---|---|---|
| Цены | цена | существительное (NOUN) |
| на | на | предлог (PREP) |
| сталь | стал


### AGENTS.md — инструкции для агента в репозитории

Есть и еще более простой стандарт, который стоит знать, если вы работаете с агентами для программирования (Codex, Cursor, GitHub Copilot, Gemini CLI и др.). [**AGENTS.md**](https://agents.md) — это «README для агентов»: обычный Markdown-файл в корне репозитория, где написано то, что вы сказали бы новому коллеге: как установить зависимости и запустить тесты, какие соглашения о коде, что нельзя трогать, где лежат данные.

- Формат появился в 2025 году как совместная инициатива OpenAI, Google, Cursor и других; с декабря 2025 года, как и MCP, развивается в Agentic AI Foundation; по данным сайта, его используют более 60 тысяч открытых проектов и больше двадцати агентов.
- Обязательных полей нет — это просто текст.
- В больших репозиториях можно класть отдельные AGENTS.md в подпапки: агент читает ближайший к файлу, с которым работает, и при противоречии ближайший важнее (в основном репозитории OpenAI таких файлов почти девяносто).

Чем это отличается от навыков: AGENTS.md агент читает **всегда**, в начале работы с проектом — поэтому туда пишут только то, что нужно почти в каждой задаче, и держат его коротким (он занимает контекст и стоит в самом начале промпта — значит, его изменения сбрасывают кеш, см. ниже). Все, что нужно лишь иногда, лучше вынести в навыки, которые загружаются по требованию. (У некоторых агентов есть свои аналоги с тем же смыслом, например `CLAUDE.md` у Claude Code.)


In [ ]:

# пример AGENTS.md для учебного проекта по корпусной лингвистике
agents_md = """# AGENTS.md

## Проект
Скрипты для разметки и анализа корпуса новостей на русском языке.

## Окружение
- Python 3.12, зависимости: `pip install -r requirements.txt`
- Тесты: `pytest tests/` (должны проходить перед каждым коммитом)

## Данные
- Сырые тексты: `data/raw/` — только чтение, ничего там не изменять
- Разметка: CoNLL-U в `data/annotated/`, тегсет Universal Dependencies

## Соглашения
- Морфология — через pymorphy3; не писать собственные эвристики для лемматизации
- Новые скрипты кладем в `scripts/` с docstring и примером запуска
- Для разметки частей речи используй навык opencorpora-pos-tagging
"""
Path("AGENTS.md").write_text(agents_md, encoding="utf-8")
print(f"AGENTS.md: {len(encoding.encode(agents_md))} токенов — они будут в контексте каждой сессии агента")


AGENTS.md: 190 токенов — они будут в контексте каждой сессии агента


## Контекстное окно и его сжатие

**Контекстное окно** — максимальное число токенов, которое модель обрабатывает за один запрос: и вход, и выход (включая рассуждения). У флагманских моделей 2026 года — около миллиона токенов. Модель ничего не помнит между запросами, поэтому все, что она должна учитывать, должно быть в контексте. «Память» агента — это целиком работа harness: что положить в окно, что выбросить, что пересказать. У агента туда входят:
- системные инструкции;
- описания **всех** инструментов (в каждом запросе!);
- вся история диалога;
- результаты инструментов (часто самая большая часть: содержимое файлов, результаты поиска);
- рассуждения модели.

Посчитаем, сколько занимает наш агент из раздела про навыки, еще до первого вопроса пользователя:

In [59]:
def n_tokens(obj):
    text = obj if isinstance(obj, str) else json.dumps(obj, ensure_ascii=False)
    return len(encoding.encode(text))


print("инструкции с каталогом навыков:", n_tokens(SKILL_INSTRUCTIONS))
print("описания инструментов:          ", n_tokens(SKILL_TOOLS))
print("текст навыка (если загружен):   ", n_tokens(skills["opencorpora-pos-tagging"]["body"]))
print("один результат morph_parse:     ", n_tokens(morph_parse("стали")))

инструкции с каталогом навыков: 84
описания инструментов:           304
текст навыка (если загружен):    173
один результат morph_parse:      188


Цифры маленькие, но у реальных агентов десятки инструментов, а результаты — целые файлы и веб-страницы; за долгую сессию контекст заполняется очень быстро.

### Больше контекста ≠ лучше

Даже если информация помещается в окно, это не значит, что модель ей хорошо пользуется:
- **«Lost in the Middle»** ([Liu et al., 2023](https://arxiv.org/abs/2307.03172)): информацию в начале и в конце контекста модели находят лучше, чем в середине;
- **RULER** ([2024](https://arxiv.org/abs/2404.06654)): из 17 моделей, заявлявших контекст от 32 тыс. токенов, лишь половина нормально работала на 32 тыс.;
- **«Context Rot»** ([Chroma, 2025](https://www.trychroma.com/research/context-rot)): на 18 моделях качество заметно падает с ростом длины входа даже на простых задачах, и даже один отвлекающий фрагмент ухудшает результат.
- **ONERULER** ([Kim et al., 2025](https://arxiv.org/abs/2503.01996)) — тест длинного контекста на 26 языках, включая русский. Английский оказался только на 6-м месте (первым — польский), разрыв между языками с большим и малым количеством данных растет с длиной контекста, а если инструкция и текст на разных языках, точность меняется до 20% в зависимости от языка инструкции.

Поэтому вместо «положим в контекст все» сейчас говорят о **context engineering** ([Anthropic, 2025](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)): контекст — ограниченный «бюджет внимания», и в нем должен быть минимальный набор действительно нужных токенов.

### Как сжимать контекст

- **Обрезка** — выкинуть самые старые сообщения (просто, но теряется информация);
- **очистка результатов инструментов** — заменить старые результаты короткими заглушками: модель уже их использовала, а сам вызов остается в истории;
- **сжатие (compaction)** — когда контекст приближается к пределу, попросить модель кратко пересказать историю и продолжить с пересказа;
- **внешняя память** — агент записывает важное в файл (заметки, план) и перечитывает при необходимости;
- **подагенты** — отдельная задача решается в отдельном «чистом» контексте, а в основной возвращается только краткий итог;
- **загрузка по требованию** — держать в контексте не содержимое, а ссылки (пути к файлам, запросы) и загружать нужное инструментами (это та же идея, что у навыков).



Провайдеры встраивают то же самое в API: в Responses API можно передать `context_management=[{"type": "compaction", "compact_threshold": 200_000}]`, и при превышении порога сервер сам сожмет историю (результат — зашифрованный элемент, который нужно передавать дальше как есть); у Anthropic есть аналогичное сжатие и автоматическая очистка старых результатов инструментов. Но у любого сжатия есть цена, которая станет понятна в следующем разделе: оно ломает кеш.

## Кеширование промпта

При обработке промпта модель вычисляет внутренние представления каждого токена (KV-кеш), и для одинакового **начала** промпта они получаются одинаковыми. Поэтому провайдеры (точнее, их серверная часть harness) сохраняют их и при следующем запросе с тем же началом пересчитывают только новую часть. Для пользователя это дешевле (кешированные входные токены у OpenAI и Anthropic стоят около 10% от обычной цены) и быстрее (меньше задержка до первого токена). У OpenAI кеширование включается автоматически для промптов от ~1024 токенов.

Ключевое слово — **начало** (префикс). Кеш работает, только если промпт **совпадает токен в токен с самого начала**. Одно изменение в начале — и весь кеш после него не используется. Отсюда главное правило, которое прямо сформулировано в [документации OpenAI](https://developers.openai.com/api/docs/guides/prompt-caching): **статичное — в начало, меняющееся — в конец**. Стабильные инструкции, описания инструментов, примеры и справочные материалы — первыми; дату, имя пользователя и прочие персональные детали — «в конце, а не в начале».

Самая частая ошибка — «вежливо» начать системный промпт с текущей даты и времени или имени пользователя. Посмотрим, что это делает с кешем: соберем три последовательных запроса двумя способами и посчитаем, сколько токенов в начале каждого совпадает с предыдущим запросом (только эта часть может быть взята из кеша).

In [ ]:
import datetime

INSTRUCTIONS = "Ты помощник лингвиста. Отвечай кратко, ссылайся на граммемы OpenCorpora. " * 40  # большая статичная часть
TOOL_DEFINITIONS = json.dumps(TOOLS, ensure_ascii=False)
EXAMPLES = "\n".join(f"Пример {i}: «слово» -> разбор с леммой и граммемами" for i in range(60))

requests_log = [("Анна", "Какой падеж у слова «стали»?"),
                ("Борис", "Разбери слово «пила»."),
                ("Вера", "Поставь «окно» в родительный падеж множественного числа.")]


def bad_prompt(user, question, now):
    return f"Сегодня {now:%Y-%m-%d %H:%M:%S}. Пользователь: {user}.\n{INSTRUCTIONS}\n{TOOL_DEFINITIONS}\n{EXAMPLES}\nВопрос: {question}"


def good_prompt(user, question, now):
    return f"{INSTRUCTIONS}\n{TOOL_DEFINITIONS}\n{EXAMPLES}\nСегодня {now:%Y-%m-%d %H:%M:%S}. Пользователь: {user}.\nВопрос: {question}"


def shared_prefix(a, b):
    ta, tb = encoding.encode(a), encoding.encode(b)
    n = 0
    while n < min(len(ta), len(tb)) and ta[n] == tb[n]:
        n += 1
    return n, len(tb)


start = datetime.datetime(2026, 10, 13, 12, 0, 0)
for name, build in [("дата и имя в начале", bad_prompt), ("дата и имя в конце ", good_prompt)]:
    prompts = [build(user, q, start + datetime.timedelta(seconds=40 * i)) for i, (user, q) in enumerate(requests_log)]
    for i in range(1, len(prompts)):
        n, total = shared_prefix(prompts[i - 1], prompts[i])
        print(f"{name}: запрос {i + 1} — можно взять из кеша {n}/{total} токенов ({n / total:.0%})")

дата и имя в начале: запрос 2 — можно взять из кеша 13/2670 токенов (0%)
дата и имя в начале: запрос 3 — можно взять из кеша 11/2679 токенов (0%)
дата и имя в конце : запрос 2 — можно взять из кеша 2651/2670 токенов (99%)
дата и имя в конце : запрос 3 — можно взять из кеша 2649/2679 токенов (99%)
